# Progetto Introduzione alla Data Science e al pensiero computazionale a.a. 2025-26 - Gruppo 8

Il Gruppo 8 è composto da:
*   **Gaia Gasparini:** matricola n. 1226091, gaia.gasparini@studio.unibo.it
*   **Aurora Paolicchi:** matricola n. 1216261, aurora.paolicchi@studio.unibo.it
*   **Lucrezia Volpato:** matricola n. 1231337, lucrezia.volpato@studio.unibo.it







## **Customer Churn Prediction**

**Obiettivo:** prevedere se un cliente abbandonerà il servizio (churn) sulla base
di informazioni demografiche, contrattuali e di utilizzo del servizio.

**Dataset:** Telco Customer Churn — 7.043 osservazioni, 21 variabili, target: `Churn` (Yes/No)

**Fasi del progetto**:
*   Fase 1: Descrizione e comprensione del dataset
*   Fase 2: Analisi esplorativa e visualizzazione
*   Fase 3: Modellazione
*   Fase 4: Valutazione e interpretazione dei risultati

# Preparazione ambiente

Prima di iniziare dobbiamo preparare l'ambiente Colab importando **Pandas**, **Seaborn** e **Scikit-learn**, le librerie Python che abbiamo visto in questo corso.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Carichiamo il dataset

In [ ]:
from pandas.core.frame import DataFrame
# Carica il dataset
df = pd.read_csv('sample_data/Customer_Churn.csv')

# Mostra le prime 5 righe per verificare che sia tutto ok
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# Fase 1: Descrizione e Composizione del dataset

## D1: Quante osservazioni e variabili contiene il dataset?



In [ ]:
# Dimensioni del dataset: numero di osservazioni e variabili
print("Dimensioni del dataset:", df.shape)
print("Il numero di osservazioni del Dataset è:", df.shape[0])
print("Il numero di variabili del Dataset è:", df.shape[1])

Dimensioni del dataset: (7043, 21)
Il numero di osservazioni del Dataset è: 7043
Il numero di variabili del Dataset è: 21


Il dataset è composto da 7.043 osservazioni e 21 variabili.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


Il dataset è composto da 21 variabili: 3 numeriche (`tenure`, `MonthlyCharges`) e 18 categoriche, inclusa la variabile target `Churn`. `SeniorCitizen`, pur essendo codificata come intero (0/1), è di natura categorica. `TotalCharges`, pur rappresentando un importo monetario, è codificata come stringa — entrambe le anomalie saranno corrette in fase di preprocessing.

## D2: Esistono valori mancati o Id cliente duplicati?

In [ ]:
# Verifica Id duplicati
id_duplicati = df.duplicated(subset=["customerID"]).sum()
print("Ci sono", id_duplicati, "id duplicati.")

Ci sono 0 id duplicati.


In [ ]:
# Verifica valori mancanti espliciti (NaN)
print("Valori NaN per colonna:")
print(df.isnull().sum())


Valori NaN per colonna:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [ ]:
# Valori vuoti nascosti in TotalCharges
print("\nValori vuoti in TotalCharges:", (df['TotalCharges'].str.strip() == '').sum())
print(df[df['TotalCharges'].str.strip() == ''][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])


Valori vuoti in TotalCharges: 11
      customerID  tenure  MonthlyCharges TotalCharges
488   4472-LVYGI       0           52.55             
753   3115-CZMZD       0           20.25             
936   5709-LVOEQ       0           80.85             
1082  4367-NUYAO       0           25.75             
1340  1371-DWPAZ       0           56.05             
3331  7644-OMVMY       0           19.85             
3826  3213-VVOLG       0           25.35             
4380  2520-SGTTA       0           20.00             
5218  2923-ARZLG       0           19.70             
6670  4075-WKNIU       0           73.35             
6754  2775-SEFEE       0           61.90             


Non sono presenti `customerID` duplicati.
Il dataset non contiene valori NaN espliciti, tuttavia `TotalCharges` nasconde 11 valori vuoti (" ") non rilevabili con `isnull()`. Tutti e 11 i casi corrispondono a clienti con `tenure = 0`, probabilmente appena acquisiti e senza addebiti registrati.

## D3: Qual è la distribuzione percentuale della variabile churn?



In [ ]:
# Distribuzione della variabile target Churn
churn_percentuale = df['Churn'].value_counts(normalize=True) * 100
print(churn_percentuale.round(2))

Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64


Il dataset presenta un **class imbalance** moderato: il 73.46% dei clienti non ha abbandonato il servizio contro il 26.54% che lo ha fatto (~3:1). Questo squilibrio richiede attenzione nella scelta delle metriche e dell'algoritmo di classificazione

## D4: Come si distribuisce la variabile 'tenure' (numero di mesi di permanenza del cliente)?


In [ ]:
# Misure di centralità (media e mediana), deviazione standard e range dei valori della variabile 'tenure'
media_tenure = df["tenure"].mean()
mediana_tenure = df["tenure"].median()
deviazione_standard_tenure = df["tenure"].std()
min_tenure = df["tenure"].min()
max_tenure = df["tenure"].max()
print (f"tenure: media = {media_tenure:.2f}, mediana = {mediana_tenure:.2f}, deviazione standard = {deviazione_standard_tenure:.2f}, min = {min_tenure:.2f}, max = {max_tenure:.2f}")


tenure: media = 32.37, mediana = 29.00, deviazione standard = 24.56, min = 0.00, max = 72.00


La `tenure` ha media pari a **32,4** **mesi** e mediana di **29 mesi**. Le due misure sono relativamente vicine, segno che la permanenza media e quella del cliente "centrale" non divergono in modo marcato. I valori vanno da un minimo di **0** a un massimo di **72 mesi**, con una deviazione standard di **24,6**: una dispersione ampia, che indica clienti molto eterogenei per anzianità, dai nuovissimi a chi è presente da sei anni.

## D5: Come si distribuiscono le variabili di spesa MonthlyCharges (canone mensile) e TotalCharges (spesa cumulata)?

In [ ]:
# Misure di centralità (media e mediana), deviazione standard e range dei valori della variabile 'Monthly Charges'
media_monthly_charges = df["MonthlyCharges"].mean()
mediana_monthly_charges = df["MonthlyCharges"].median()
deviazione_standard_monthly_charges = df["MonthlyCharges"].std()
min_monthly_charges = df["MonthlyCharges"].min()
max_monthly_charges = df["MonthlyCharges"].max()

# La variabile 'Total Charges' viene letta da pandas come testo e non come numero a causa degli 11 valori vuoti bisogna quindi prima convertirla.

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# errors=coerce dice "se trovi un valore che non riesci a convertire (lo spazio vuoto), non bloccarti: trasformalo in NaN"
# Adesso possiamo procedere con i calcoli di media, mediana, deviazione standard e range dei valori
media_total_charges = df["TotalCharges"].mean()
mediana_total_charges = df["TotalCharges"].median()
deviazione_standard_total_charges = df["TotalCharges"].std()
min_total_charges = df["TotalCharges"].min()
max_total_charges = df["TotalCharges"].max()

print(f"MonthlyCharges: media = {media_monthly_charges:.2f}, mediana = {mediana_monthly_charges:.2f}, std = {deviazione_standard_monthly_charges:.2f}, min = {min_monthly_charges:.2f}, max = {max_monthly_charges:.2f}")
print(f"TotalCharges: media = {media_total_charges:.2f}, mediana = {mediana_total_charges:.2f}, std = {deviazione_standard_total_charges:.2f}, min = {min_total_charges:.2f}, max = {max_total_charges:.2f}")



MonthlyCharges: media = 64.76, mediana = 70.35, std = 30.09, min = 18.25, max = 118.75
TotalCharges: media = 2283.30, mediana = 1397.47, std = 2266.77, min = 18.80, max = 8684.80


Per `MonthlyCharges` la media è **64,8** e la mediana **70,4**: la media leggermente inferiore alla mediana suggerisce una possibile asimmetria. I valori si distribuiscono tra **18,3** e **118,8**, con deviazione standard di **30,1**.

Per `TotalCharges` lo scarto è molto più ampio (media **2.283,3** contro mediana **1.397,5**): la media nettamente superiore alla mediana indica che pochi clienti con spesa cumulata elevata alzano il valore medio. Lo confermano l'intervallo molto esteso (da **18,8** a **8.684,8**) e una deviazione standard di **2.266,8**, quasi pari alla media stessa, segnale di una variabilità molto marcata. Questa è un'ipotesi sulla forma delle distribuzioni che verificheremo nella fase di analisi esplorativa.